# 🔧 Fine-Tuning Gemma 4 on Consumer Hardware
### Live Workshop Notebook — Why Prompting Alone Is Not Enough

Welcome! This notebook is built to survive a **live workshop with a room full of people
running it at the same time on free Colab T4 GPUs.** Read this cell before you touch anything.

## ⚠️ Read this first — the 5 rules that prevent 90% of live errors

1. **Run cells top to bottom, once each.** Don't skip around. If something errors, re-run
   from the "🔁 Recovery" cell in Section 0.4, not from a random cell in the middle.
2. **After the install cell (0.2), you MUST restart the runtime.** Colab will show a button
   — click it. Skipping this is the #1 cause of `ImportError` / version-mismatch crashes.
3. **Default model is Gemma 4 E2B (2B params) in 4-bit.** This fits comfortably in a free T4
   (~15GB VRAM). Don't switch to E4B or 12B unless you have Colab Pro — we'll show you how,
   but it's opt-in, not default, specifically so the workshop doesn't collapse into a wave of
   `CUDA out of memory` errors.
4. **If you ever see `CUDA out of memory`:** go to `Runtime → Restart session`, then re-run
   from Section 0 onward. Do NOT just re-run the training cell — the GPU memory is already
   fragmented and it will error again.
5. **Training is capped at a small step count on purpose.** This is a workshop, not a full
   training run. You will see the loss drop and get a working adapter in ~3–5 minutes. The
   "go further at home" section at the end shows you how to scale it up later.

## What you'll build
By the end of this notebook you'll have fine-tuned Gemma 4 on a small instruction dataset,
merged the LoRA adapter, and run a before/after comparison proving the model learned
something new — using nothing but a free Colab GPU.

## Agenda
| Section | What happens |
|---|---|
| 0 | Environment setup, GPU check, installs, HF login |
| 1 | Load Gemma 4 in 4-bit, sanity-check it responds |
| 2 | Prepare a small dataset in Gemma's chat format |
| 3 | Configure LoRA and train (fast, capped for live demo) |
| 4 | Merge adapter, run before/after comparison |
| 5 | 🧪 Bring-your-own-data challenge |
| 6 | Troubleshooting cheat-sheet + where to go next |


---
## 0.1 — Check your GPU

Run this first. If it errors or shows no GPU, go to **Runtime → Change runtime type → T4 GPU**,
save, and re-run this cell before continuing.

In [1]:
import os

# Must be set BEFORE torch is ever imported anywhere in this session, so it's set here,
# first, before anything else touches CUDA. Reduces memory fragmentation, which is a real
# contributor to "OutOfMemoryError" even when nvidia-smi shows free memory available.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode != 0:
    print("❌ No GPU detected.")
    print("Fix: Runtime menu -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save")
    print("Then re-run this cell.")
else:
    print("✅ GPU detected:\n")
    # Print just the GPU name + memory line, not the whole noisy table
    for line in result.stdout.splitlines():
        if "MiB" in line or "Tesla" in line or "T4" in line or "A100" in line or "L4" in line:
            print(line)
    print("\nIf you see 'T4', you're on the free tier default -> use E2B (default in this notebook).")
    print("If you see 'A100' or 'L4', you're on Colab Pro -> you may optionally try E4B in Section 0.3.")


✅ GPU detected:

|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |

If you see 'T4', you're on the free tier default -> use E2B (default in this notebook).
If you see 'A100' or 'L4', you're on Colab Pro -> you may optionally try E4B in Section 0.3.


---
## 0.2 — Install dependencies

This installs pinned, tested versions. **After this cell finishes, restart the runtime**
(`Runtime → Restart session`, or click the button Colab pops up), then continue from
Section 0.3. Do not re-run this install cell after restarting — just move on.

Why restart? `transformers` and `torch` register themselves at import time. If they were
already imported anywhere in this session before the upgrade, Python keeps using the old
cached version in memory, causing confusing `AttributeError`s that have nothing to do with
your code.

**Important:** we deliberately do **not** upgrade `torch` itself. Colab's preinstalled
`torch` is already a CUDA-enabled build that's version-matched with the preinstalled
`torchvision`/`torchaudio`. If you `pip install -U torch` on its own, it pulls in a newer
`torch` while leaving the old `torchvision` behind — the two no longer agree on their C++
extension ABI, and you get a confusing `RuntimeError: operator torchvision::nms does not
exist`, which then cascades into unrelated-looking import errors elsewhere (like
`transformers` failing to load `Gemma4Config`). Since this workshop is text-only, we sidestep
the whole problem by removing `torchvision`/`torchaudio` instead of trying to keep them in
sync.

In [2]:
%%capture install_log
import sys

# Remove torchvision/torchaudio: we don't need them for text-only fine-tuning, and keeping
# them around risks a torch/torchvision ABI mismatch (RuntimeError: torchvision::nms) that
# cascades into confusing transformers import errors later. We also deliberately do NOT
# upgrade torch itself -- Colab's preinstalled torch is already the right CUDA-matched build.
get_ipython().system("pip uninstall -y -q torchvision torchaudio")

# Pinned versions chosen for Gemma 4 compatibility + Colab T4 stability.
packages = [
    "transformers>=5.10.1",
    "datasets",
    "accelerate",
    "evaluate",
    "bitsandbytes",
    "trl",
    "peft>=0.19.0",
    "protobuf",
    "sentencepiece",
    "hf_transfer",
]

get_ipython().system(f"pip install -q -U {' '.join(packages)}")
print("DONE")


In [3]:
# Show the tail of the install log so you can spot real errors
# (pip prints a lot of harmless dependency-resolution noise — look for the word "ERROR")
log_text = install_log.stdout if 'install_log' in dir() else ""
# Pip prints "ERROR:" for dependency-resolver noise that is usually harmless
error_lines = [
    l for l in log_text.splitlines()
    if "ERROR" in l and "dependency resolver" not in l.lower()
]

if error_lines:
    print("⚠️ Possible install issues found:")
    for l in error_lines:
        print(" -", l)
    print("\nIf you see a real ERROR above (not just a warning), re-run the install cell once.")
    print("If it persists, Runtime -> Restart session, then run install cell again.")
else:
    print("✅ Installs completed with no ERROR lines.")

print("\n➡️ NOW: Runtime menu -> Restart session (or click the restart button Colab shows).")
print("➡️ After restarting, continue from the next cell (0.3). Do NOT re-run the install cell.")


✅ Installs completed with no ERROR lines.

➡️ NOW: Runtime menu -> Restart session (or click the restart button Colab shows).
➡️ After restarting, continue from the next cell (0.3). Do NOT re-run the install cell.


---
## 0.3 — Configuration + Hugging Face login

**Before running this cell:**
1. Accept the Gemma 4 license at https://huggingface.co/google/gemma-4-E2B-it (one click, "Agree").
2. Get a token at https://huggingface.co/settings/tokens (read access is enough).
3. In Colab, click the 🔑 key icon in the left sidebar → add a secret named `HF_TOKEN` →
   paste your token → toggle "Notebook access" on.

This cell will tell you clearly if any of that is missing instead of failing with a cryptic
error later.

In [5]:
# Preflight: catch a torch/torchvision ABI mismatch here, in 2 seconds, instead of
# 5 minutes from now buried inside a Gemma4Config import error.
import torch

print(f"torch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

try:
    import torchvision  # noqa: F401
    print("⚠️ torchvision is still installed. If Section 1 later throws")
    print("   'RuntimeError: operator torchvision::nms does not exist', run:")
    print("   !pip uninstall -y -q torchvision torchaudio")
    print("   then Runtime -> Restart session, and continue from here again.")
except ImportError:
    print("✅ torchvision removed cleanly (expected) -- no ABI mismatch risk.")

if not torch.cuda.is_available():
    print("❌ No GPU visible to torch. Runtime -> Change runtime type -> T4 GPU, then restart.")


torch version: 2.11.0+cu128 | CUDA available: True
✅ torchvision removed cleanly (expected) -- no ABI mismatch risk.


In [6]:
import os

# ── Model tier picker ────────────────────────────────────────────────────
# Change this dropdown value if you're on Colab Pro with an A100/L4.
# For the live workshop on free T4, leave this as "E2B".
MODEL_TIER = "E2B"  # @param ["E2B", "E4B", "12B"]

MODEL_MAP = {
    "E2B": "google/gemma-4-E2B-it",   # safest default: fits free T4 in 4-bit with headroom
    "E4B": "google/gemma-4-E4B-it",   # needs QLoRA 4-bit even on T4; tighter fit
    "12B": "google/gemma-4-12B-it",   # requires A100/L4 (Colab Pro) — will likely OOM on free T4
}
MODEL_ID = MODEL_MAP[MODEL_TIER]

if MODEL_TIER == "12B":
    print("⚠️ You selected the 12B model. This needs an A100/L4 GPU (Colab Pro/Pro+).")
    print("   On a free T4 this will almost certainly hit CUDA out of memory.")
    print("   If you're on free Colab, change MODEL_TIER back to 'E2B' above and re-run.")

print(f"Selected model: {MODEL_ID}")

# ── Hugging Face auth ────────────────────────────────────────────────────
from huggingface_hub import login, HfApi

# Priority: env var (local/CLI) -> Colab secret -> paste below for one-off local runs
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not hf_token:
    raise RuntimeError(
        "No HF_TOKEN found.\n"
        "Fix: click the key icon in the left sidebar, add a secret named exactly 'HF_TOKEN', "
        "paste a token from https://huggingface.co/settings/tokens, enable notebook access, "
        "then re-run this cell."
    )

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # faster, more reliable downloads on shared wifi

# Confirm gated access BEFORE we get deep into loading, so the error is clear and early
try:
    HfApi().model_info(MODEL_ID, token=hf_token)
    print("✅ Hugging Face auth OK, and you have access to", MODEL_ID)
except Exception as e:
    raise RuntimeError(
        f"Could not access {MODEL_ID}. Most likely cause: you haven't accepted the license yet.\n"
        f"Fix: open https://huggingface.co/{MODEL_ID}, click 'Agree and access repository', "
        f"then re-run this cell.\nOriginal error: {e}"
    )


Selected model: google/gemma-4-E2B-it
✅ Hugging Face auth OK, and you have access to google/gemma-4-E2B-it


---
## 0.4 — 🔁 Recovery cell (use this if anything crashes mid-workshop)

If a later cell throws a CUDA / memory / state error, run this cell to clear the GPU, then
**re-run from Section 1 onward** (you don't need to redo installs or login unless you
restarted the runtime).

In [7]:
import gc, torch

for name in ["model", "trainer", "merged_model"]:
    if name in dir():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    free, total = torch.cuda.mem_get_info()
    print(f"✅ GPU memory cleared. Free: {free/1e9:.1f} GB / {total/1e9:.1f} GB")
else:
    print("⚠️ No GPU visible right now — check Section 0.1 again.")


✅ GPU memory cleared. Free: 15.5 GB / 15.6 GB


---
## 1 — Load Gemma 4 in 4-bit

We load with `BitsAndBytesConfig` 4-bit quantization. This is the single biggest lever for
"won't fry your machine" — it's the difference between ~5GB and ~17GB of VRAM for E4B.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, model_max_length=512)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    print("✅ Model loaded:", MODEL_ID)
except torch.cuda.OutOfMemoryError:
    print("❌ CUDA out of memory while loading the base model.")
    print("Fix: Runtime -> Restart session, run the Recovery cell (0.4) after restart is not needed,")
    print("     just re-run from Section 0.3, and make sure MODEL_TIER = 'E2B'.")
    raise
except Exception as e:
    err_text = str(e)
    print("❌ Model load failed:", e)
    if "torchvision" in err_text or "Gemma4Config" in err_text:
        print("\nThis specific error means torch/torchvision are out of sync (an ABI mismatch),")
        print("which confusingly surfaces as a Gemma4Config import failure. Fix:")
        print("  1. Run: !pip uninstall -y -q torchvision torchaudio")
        print("  2. Runtime -> Restart session")
        print("  3. Continue from Section 0.3 onward (skip the install cell, it already ran).")
    else:
        print("Common causes: license not accepted (see 0.3), or runtime wasn't restarted after installs.")
    raise

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

### 1.1 — Sanity check: does it respond at all?

Quick smoke test before we invest time in training. If this cell hangs for more than ~30s
or errors, stop and use the Recovery cell (0.4).

In [ ]:
from transformers import TextStreamer

def ask(prompt, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt}]
    # Apply chat template, which returns a dictionary-like object (BatchEncoding)
    encoded_inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )
    # Extract input_ids and move to device
    input_ids = encoded_inputs["input_ids"].to(model.device)
    # If attention_mask is present and needed, extract and move it as well
    # attention_mask = encoded_inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids, # Pass the input_ids tensor directly
            # attention_mask=attention_mask, # Pass attention_mask if needed
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Slice the output using the length of the input_ids tensor
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

print(ask("In one sentence, what is LoRA fine-tuning?"))

NameError: name 'tokenizer' is not defined

In [ ]:
# Free the memory used by the sanity-check generation above before we move into
# training setup -- KV cache and activation buffers from .generate() don't always
# release themselves immediately, and that's memory we need back for Section 3.
import gc

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free/1e9:.1f} GB / {total/1e9:.1f} GB")
if free / total < 0.35:
    print("⚠️ Less than ~35% free. If Section 3 OOMs, run the Recovery cell (0.4),")
    print("   then re-run from Section 1 (reload the model) before trying again.")


GPU memory free: 8.7 GB / 15.6 GB


---
## 2 — Prepare the dataset

For the live demo we use a small built-in sample dataset so everyone gets a working result
in minutes, no matter their upload speed on venue wifi. Section 5 shows you how to swap in
your own data afterward.

In [ ]:
from datasets import Dataset

# Small demo dataset: teaching the model a specific formatting/persona quirk so the
# before/after comparison is obvious. Feel free to edit these examples live.
demo_examples = [
    {"instruction": "What is the capital of France?",
     "response": "📍 Capital: Paris. (Answered in Workshop-Bot house style.)"},
    {"instruction": "Explain what a neural network is.",
     "response": "📍 Concept: A neural network is layers of connected nodes that learn patterns from data. (Answered in Workshop-Bot house style.)"},
    {"instruction": "What's 2 + 2?",
     "response": "📍 Answer: 4. (Answered in Workshop-Bot house style.)"},
    {"instruction": "Name a popular programming language.",
     "response": "📍 Answer: Python. (Answered in Workshop-Bot house style.)"},
    {"instruction": "What does GPU stand for?",
     "response": "📍 Answer: Graphics Processing Unit. (Answered in Workshop-Bot house style.)"},
    {"instruction": "Give a one-word synonym for 'happy'.",
     "response": "📍 Answer: Joyful. (Answered in Workshop-Bot house style.)"},
] * 6  # repeated to give the tiny demo run enough steps to visibly converge

def to_chat_format(example):
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "model", "content": example["response"]},
        ]
    }

raw_dataset = Dataset.from_list(demo_examples).map(to_chat_format)
split = raw_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset, eval_dataset = split["train"], split["test"]

print(f"Train examples: {len(train_dataset)} | Eval examples: {len(eval_dataset)}")
print("\nSample formatted example:")
print(train_dataset[0])


Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Train examples: 30 | Eval examples: 6

Sample formatted example:
{'instruction': "Give a one-word synonym for 'happy'.", 'response': '📍 Answer: Joyful. (Answered in Workshop-Bot house style.)', 'messages': [{'role': 'user', 'content': "Give a one-word synonym for 'happy'."}, {'role': 'model', 'content': '📍 Answer: Joyful. (Answered in Workshop-Bot house style.)'}]}


---
## 3 — Configure LoRA and train

Steps are deliberately capped (`max_steps`) so this finishes in ~3–5 minutes live, even with
a room full of people hitting the shared model download cache at once. This is enough to see
the loss curve move and the model pick up the new style — not enough for a production
fine-tune. Section 5 shows how to scale this up at home.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import gc
from transformers import DataCollatorForLanguageModeling

def prepare_model_for_lora_training(model):
    '''Lightweight stand-in for peft's prepare_model_for_kbit_training.

    That library function upcasts EVERY non-4bit parameter to fp32 in one shot --
    which includes the embedding and lm_head layers, not just norm layers. For a
    model with a ~256k-token vocab (Gemma), that embedding/lm_head upcast alone is
    a multi-GB one-time allocation, which is exactly what was OOMing here: the
    error said 'Tried to allocate 8.75 GiB' for what's described as upcasting
    'a few layers' -- that 8.75 GiB is the embedding + lm_head, not the norms.

    We only need fp32 precision on the norm layers for training stability;
    embeddings/lm_head are frozen lookup tables here and are fine to leave at
    bf16, so we skip upcasting them and avoid the spike entirely.
    '''
    for param in model.parameters():
        param.requires_grad = False

    upcast_count = 0
    for name, module in model.named_modules():
        if "norm" in module.__class__.__name__.lower():
            module.to(torch.float32)
            upcast_count += 1

    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

    print(f"✅ Prepared for training: {upcast_count} norm layers upcast to fp32 "
          f"(embeddings/lm_head left at bf16 -- this is what avoids the OOM).")
    return model

# One more clear-out immediately before the upcast step below.
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free before prep: {free/1e9:.1f} GB / {total/1e9:.1f} GB")

try:
    model = prepare_model_for_lora_training(model)
except torch.cuda.OutOfMemoryError:
    print("❌ CUDA out of memory while preparing the model for training.")
    print("Fix: run the Recovery cell (0.4), then re-run from Section 1 to reload a fresh")
    print("     model, then come straight back to this cell (skip re-running 1.1's sanity check).")
    raise

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="./gemma4-workshop-adapter",
    per_device_train_batch_size=1,       # kept low -- T4 free tier has ~14.5GB total
    gradient_accumulation_steps=8,       # same effective batch size (8) via accumulation instead
    max_steps=20,                # capped for a live, time-boxed workshop
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    bf16=True,
    logging_steps=5,
    save_strategy="no",          # don't burn disk/time on checkpoints during the demo
    report_to="none",
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("✅ Trainer configured. Estimated time on T4: ~4-6 minutes for 40 steps.")

GPU memory free before prep: 8.7 GB / 15.6 GB
✅ Prepared for training: 466 norm layers upcast to fp32 (embeddings/lm_head left at bf16 -- this is what avoids the OOM).


Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

✅ Trainer configured. Estimated time on T4: ~4-6 minutes for 40 steps.


In [ ]:
try:
    train_result = trainer.train()
    print("\n✅ Training complete.")
    print(f"Final training loss: {train_result.training_loss:.4f}")
except torch.cuda.OutOfMemoryError:
    print("❌ CUDA out of memory during training.")
    print("Fix: Runtime -> Restart session -> re-run from Section 0.3 with MODEL_TIER = 'E2B',")
    print("     and if it still OOMs, lower per_device_train_batch_size to 1 in the cell above.")
    raise


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
5,8.721082
10,1.716722
15,0.191876
20,0.085090



✅ Training complete.
Final training loss: 2.6787


---
## 4 — Merge the adapter and compare before vs. after

This bakes the LoRA weights into the base model so it behaves like a single fine-tuned
model, then runs the same prompts through both the original behavior (we captured this
mentally in Section 1) and the fine-tuned model side by side.

In [ ]:
ADAPTER_DIR = "./gemma4-workshop-adapter"
SAVE_DIR = "./gemma4-workshop-finetuned"

# Save the LoRA adapter before merge (small; reloadable with PeftModel)
trainer.model.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapter saved to {ADAPTER_DIR}")

merged_model = trainer.model.merge_and_unload()
merged_model.eval()
print("✅ Adapter merged into base weights.")

# Save the full merged model so Section 4.1 can reload it in bf16
merged_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ Merged model saved to {SAVE_DIR}")


✅ LoRA adapter saved to ./gemma4-workshop-adapter


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Adapter merged into base weights.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved to ./gemma4-workshop-finetuned


In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM
import os

ADAPTER_DIR = "./gemma4-workshop-adapter"
SAVE_DIR = "./gemma4-workshop-finetuned"

is_adapter = os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json"))
is_merged = os.path.exists(os.path.join(SAVE_DIR, "config.json"))

if not is_adapter and not is_merged:
    raise RuntimeError(
        f"No saved weights found. Run the merge+save cell above first.\n"
        f"Expected LoRA adapter at {ADAPTER_DIR} or merged model at {SAVE_DIR}."
    )

artifact_type = "full merged model" if is_merged else ("LoRA adapter" if is_adapter else "unknown")
print(f"Loading: {artifact_type}")

# Free training objects to make room for bf16 inference reload
for name in ["model", "trainer", "merged_model", "challenge_base", "challenge_merged", "inference_model", "inference_base"]:
    if name in dir():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

if is_merged:
    inference_model = AutoModelForCausalLM.from_pretrained(
        SAVE_DIR, torch_dtype=torch.bfloat16, device_map="auto",
    )
elif is_adapter:
    from peft import PeftModel
    inference_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
    )
    inference_model = PeftModel.from_pretrained(inference_base, ADAPTER_DIR)

inference_model.eval()
print("✅ Fine-tuned model reloaded in bf16 for stable inference.")

def ask_after_finetuning(prompt, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(inference_model.device)
    with torch.no_grad():
        output_ids = inference_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            num_beams=1, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

test_prompts = [
    "What is the capital of Germany?",
    "What's 5 + 7?",
    "Name a popular cloud provider.",
]

print("\nBEFORE (base model style) vs AFTER (fine-tuned style):\n")
for p in test_prompts:
    print(f"Prompt: {p}")
    try:
        response = ask_after_finetuning(p)
        print(f"  After fine-tuning: {response}")
    except Exception as e:
        print(f"  Error: {type(e).__name__}: {e}")
    print()

print("Look for the '📍' house-style formatting the model picked up!")

Loading: full merged model


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

✅ Fine-tuned model reloaded in bf16 for stable inference.

BEFORE (base model style) vs AFTER (fine-tuned style):

Prompt: What is the capital of Germany?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  After fine-tuning: The capital of Germany is **Berlin**.

Prompt: What's 5 + 7?
  After fine-tuning: 5 + 7 = **12**

Prompt: Name a popular cloud provider.
  After fine-tuning: A popular cloud provider is **Amazon Web Services (AWS)**.

Other popular options include **Microsoft Azure** and **Google Cloud Platform (GCP)**.

Look for the '📍' house-style formatting the model picked up!


In [ ]:
# Already saved in the cell above; this cell is optional extras
SAVE_DIR = "./gemma4-workshop-finetuned"
print(f"Your fine-tuned model is at {SAVE_DIR}")
print("To take this home: zip that folder, or push it to the Hugging Face Hub:")
print("  merged_model.push_to_hub('your-username/gemma4-workshop-finetuned')")
print("For local/offline serving, convert to GGUF and load with Ollama or llama.cpp.")


Your fine-tuned model is at ./gemma4-workshop-finetuned
To take this home: zip that folder, or push it to the Hugging Face Hub:
  merged_model.push_to_hub('your-username/gemma4-workshop-finetuned')
For local/offline serving, convert to GGUF and load with Ollama or llama.cpp.


---
## 5 — 🧪 Bring-your-own-data challenge

Now it's your turn. Swap in your own small dataset and prove the model learned *your*
pattern. Acceptance criteria: run the before/after comparison at the end of this section
and show at least one prompt where the fine-tuned output is visibly different in a way that
matches your data.

**Steps:**
1. Edit `my_examples` below with 5-15 of your own instruction/response pairs.
2. Run the cell — it reuses the same LoRA config and trainer setup from Section 3.
3. Compare outputs at the bottom.

Keep it small (under ~20 examples, short responses) so it still trains in a couple of
minutes on shared workshop wifi and GPU.

In [ ]:
pip install -U trl peft transformers accelerate bitsandbytes

In [ ]:
import gc
import torch

# Delete everything related to training
for var in list(globals().keys()):
    if any(k in var.lower() for k in [
        "model", "trainer", "optimizer", "scheduler",
        "challenge", "merged", "peft", "lora"
    ]):
        try:
            del globals()[var]
        except:
            pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
print("Reserved :", torch.cuda.memory_reserved()/1024**3, "GB")

In [ ]:
# ── EDIT ME ──────────────────────────────────────────────────────────────
my_examples = [
        {
        "instruction": "What is machine learning?",
        "response": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed for every task."
    },
    {
        "instruction": "Explain overfitting.",
        "response": "Overfitting occurs when a model learns the training data too closely, including noise, causing poor performance on unseen data."
    },
    {
        "instruction": "What is supervised learning?",
        "response": "Supervised learning is a type of machine learning where a model is trained using labeled examples consisting of inputs and their correct outputs."
    },
    {
        "instruction": "What is the purpose of a validation set?",
        "response": "A validation set is used during training to evaluate model performance and tune hyperparameters without using the test set."
    },
    {
        "instruction": "What is gradient descent?",
        "response": "Gradient descent is an optimization algorithm that updates model parameters by moving them in the direction that minimizes the loss function."
    },
    {
        "instruction": "Why is data preprocessing important?",
        "response": "Data preprocessing improves data quality by handling missing values, scaling features, and encoding categorical variables, leading to better model performance."
    }

]
# ─────────────────────────────────────────────────────────────────────────

if len(my_examples) < 4:
    print("⚠️ Add at least 4 examples above for the training step count to make sense, then re-run.")
else:
    my_dataset = Dataset.from_list(my_examples * 6).map(to_chat_format)
    my_split = my_dataset.train_test_split(test_size=0.15, seed=42)

    my_training_args = SFTConfig(
        output_dir="./gemma4-challenge-adapter",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        max_steps=30,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=2,
        bf16=True,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        packing=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    # Reload a clean base model for the challenge so it isn't stacked on top of Section 3's adapter
    import gc
    for name in ["model", "trainer", "merged_model"]:
        if name in dir():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()

    challenge_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16,
    )
    challenge_base = prepare_model_for_lora_training(challenge_base)

    challenge_trainer = SFTTrainer(
        model=challenge_base,
        args=my_training_args,
        train_dataset=my_split["train"],
        eval_dataset=my_split["test"],
        peft_config=lora_config,
        data_collator=data_collator,
        processing_class=tokenizer,
    )

    try:
        challenge_trainer.train()
        print("✅ Challenge training complete.")
    except torch.cuda.OutOfMemoryError:
        print("❌ Out of memory. Free some VRAM: run the Recovery cell (0.4), then re-run this cell.")
        raise


In [ ]:
if "challenge_trainer" in dir():
    challenge_merged = challenge_trainer.model.merge_and_unload()
    challenge_merged.eval()

    def ask_challenge(prompt, max_new_tokens=80):
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        ).to(challenge_merged.device)
        with torch.no_grad():
            out = challenge_merged.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True)

    print("Your fine-tuned model's response to one of your own training prompts:\n")
    demo_prompt = my_examples[0]["instruction"]
    print(f"Prompt: {demo_prompt}")
    print(f"Response: {ask_challenge(demo_prompt)}")
else:
    print("Run the training cell above first (with at least 4 examples in my_examples).")


---
## 6 — Troubleshooting cheat-sheet

| Symptom | Likely cause | Fix |
|---|---|---|
| `ImportError` / `AttributeError` right after installs | Runtime wasn't restarted after Section 0.2 | `Runtime → Restart session`, then continue from 0.3 (don't re-run installs) |
| `RuntimeError: operator torchvision::nms does not exist`, or `ModuleNotFoundError: Could not import module 'Gemma4Config'` | torch/torchvision version mismatch (torchvision left behind after a torch upgrade) | `!pip uninstall -y -q torchvision torchaudio`, then `Runtime → Restart session`, then continue from 0.3 |
| `CUDA out of memory` inside the training-prep step (`Tried to allocate 8.7X GiB`) | `prepare_model_for_kbit_training` upcasting the embedding + lm_head layers (large due to Gemma's ~256k vocab) to fp32 in one shot, not just norm layers | Already fixed in this notebook — it uses a lightweight prep function that only upcasts norm layers. If you still see this, run Recovery (0.4) and reload from Section 1 |
| `CUDA out of memory` | Model too big for the GPU tier, or GPU memory fragmented from a previous run | Run the Recovery cell (0.4), confirm `MODEL_TIER = "E2B"`, re-run from 0.3 |
| Gated repo / 403 error loading the model | Gemma 4 license not accepted on that HF account | Visit the model page on huggingface.co and click "Agree and access repository" |
| `HF_TOKEN` not found | Secret not added or notebook access not toggled on | Key icon in sidebar → add `HF_TOKEN` → toggle notebook access on |
| Training cell hangs for a long time | Normal on first run — model/dataset caching to disk | Give it ~1-2 minutes before assuming it's stuck; check `nvidia-smi` in a new cell shows GPU usage |
| Everything suddenly breaks mid-workshop | Session disconnected (idle timeout, venue wifi drop) | `Runtime → Reconnect`, then `Runtime → Restart session`, then run cells top to bottom again |
| Very slow model download | Shared venue wifi + everyone downloading at once | `hf_transfer` is already enabled above to help; be patient, or use E2B (smallest) |

## Go further at home
- Swap the demo dataset for a real one (hundreds–thousands of examples).
- Raise `max_steps` from 40 to a few hundred/thousand and add an eval loop.
- Try `unsloth` for 50-80% less VRAM and faster training on the same GPU.
- Export the merged model to GGUF and serve it locally with Ollama or llama.cpp.
- Push your adapter to the Hugging Face Hub with `model.push_to_hub("your-username/name")`.

**Thanks for coming to the workshop — "Why Prompting Alone Is Not Enough."**
